<a href="https://colab.research.google.com/github/Adithya0805/AI-Fraud-Detection-System/blob/main/%22Fraud_Security_Simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from google.colab import files
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [4]:
# Create a dedicated project folder in your Drive
import os
os.makedirs('/content/drive/MyDrive/Fraud_Detection_Project', exist_ok=True)

# Download the dataset directly to your Drive folder
!kaggle competitions download -c ieee-fraud-detection -p /content/drive/MyDrive/Fraud_Detection_Project

# Unzip the raw CSV files for use in Phase 2
!unzip -q /content/drive/MyDrive/Fraud_Detection_Project/ieee-fraud-detection.zip -d /content/drive/MyDrive/Fraud_Detection_Project/

100% 118M/118M [00:00<00:00, 167MB/s]



In [5]:
import pandas as pd

# 1. Load the raw datasets from your Drive
print("Loading transaction data...")
train_transaction = pd.read_csv('/content/drive/MyDrive/Fraud_Detection_Project/train_transaction.csv')

print("Loading identity data...")
train_identity = pd.read_csv('/content/drive/MyDrive/Fraud_Detection_Project/train_identity.csv')

# 2. Merge them into a single master dataset
print("Merging datasets...")
train_data = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')

# 3. Verify the final shape (rows, columns)
print(f"Merge Complete! Final Dataset Shape: {train_data.shape}")
train_data.head()

Loading transaction data...
Loading identity data...
Merging datasets...
Merge Complete! Final Dataset Shape: (590540, 434)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [6]:
import numpy as np

def reduce_mem_usage(df):
    """Iterate through all the columns of a dataframe and modify the data type
    to reduce memory usage."""
    start_mem = df.memory_usage().sum() / 1024**2
    print(f'Initial memory usage of dataframe is {start_mem:.2f} MB')

    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            # Convert text columns (objects) to categories to save space
            df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Memory usage after optimization is: {end_mem:.2f} MB')
    print(f'Decreased RAM usage by {100 * (start_mem - end_mem) / start_mem:.1f}%!')
    return df

# Shrink our dataset
train_data = reduce_mem_usage(train_data)

Initial memory usage of dataframe is 1955.37 MB


/tmp/ipykernel_1302/2125217339.py:25: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_1302/2125217339.py:25: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_1302/2125217339.py:25: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_1302/2125217339.py:25: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_1302/2125217339.py:25: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_1302/2125217339.py:25: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_1302/2125217339.py:25: RuntimeW

Memory usage after optimization is: 525.55 MB
Decreased RAM usage by 73.1%!


In [7]:
# Set our threshold: If more than 80% of a column is missing, drop it.
missing_threshold = 0.80

# Calculate the percentage of missing values per column
missing_percentages = train_data.isnull().mean()
columns_to_drop = missing_percentages[missing_percentages > missing_threshold].index

print(f"Dropping {len(columns_to_drop)} columns that are mostly empty...")

# Execute the drop
train_data_cleaned = train_data.drop(columns=columns_to_drop)

print(f"Phase 2 Complete! New streamlined dataset shape: {train_data_cleaned.shape}")

Dropping 74 columns that are mostly empty...
Phase 2 Complete! New streamlined dataset shape: (590540, 360)


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

print("1. Encoding text columns into numbers...")
for col in train_data_cleaned.columns:
    if train_data_cleaned[col].dtype.name == 'category' or train_data_cleaned[col].dtype == 'object':
        le = LabelEncoder()
        # Convert to string to handle any lingering weird formats safely
        train_data_cleaned[col] = le.fit_transform(train_data_cleaned[col].astype(str))

print("2. Splitting data into Features (X) and Target (y)...")
# 'isFraud' is our answer key. We drop ID and Time as they aren't predictive behaviors.
X = train_data_cleaned.drop(['isFraud', 'TransactionID', 'TransactionDT'], axis=1)
y = train_data_cleaned['isFraud']

# Split 80% for training, 20% for testing.
# 'stratify=y' ensures the 1% of fraud cases are split evenly between train and test.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Data Prep Complete! Training on {X_train.shape[0]} rows and {X_train.shape[1]} features.")

1. Encoding text columns into numbers...
2. Splitting data into Features (X) and Target (y)...
Data Prep Complete! Training on 472432 rows and 357 features.


In [9]:
# Calculate the exact imbalance ratio to pass to the model
imbalance_ratio = (len(y_train) - sum(y_train)) / sum(y_train)
print(f"Imbalance Ratio (Legit/Fraud): {imbalance_ratio:.2f}")

print("Initializing XGBoost on T4 GPU...")
xgb_model = xgb.XGBClassifier(
    n_estimators=500,        # Build 500 decision trees
    max_depth=7,             # Maximum depth of each tree
    learning_rate=0.05,      # How aggressively it updates its logic
    scale_pos_weight=imbalance_ratio, # The magic bullet for imbalanced data
    tree_method='hist',      # Optimized tree building algorithm
    device='cuda',           # Force it to use the T4 GPU!
    objective='binary:logistic',
    eval_metric='aucpr',     # Area Under Precision-Recall Curve (Best for fraud)
    random_state=42
)

print("Training model... (This will take 2-4 minutes. Watch the output!)")
# We feed it the training data, and use the test data to validate its progress
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50 # Print an update every 50 trees
)

print("\n🚀 Training Complete!")

Imbalance Ratio (Legit/Fraud): 27.58
Initializing XGBoost on T4 GPU...
Training model... (This will take 2-4 minutes. Watch the output!)
[0]	validation_0-aucpr:0.36631
[50]	validation_0-aucpr:0.53510
[100]	validation_0-aucpr:0.59140
[150]	validation_0-aucpr:0.62422
[200]	validation_0-aucpr:0.64644
[250]	validation_0-aucpr:0.66426
[300]	validation_0-aucpr:0.67662
[350]	validation_0-aucpr:0.69107
[400]	validation_0-aucpr:0.70015
[450]	validation_0-aucpr:0.70971
[499]	validation_0-aucpr:0.71969

🚀 Training Complete!


In [10]:
from sklearn.metrics import classification_report, confusion_matrix
import joblib

# 1. Generate predictions on the test set
y_pred = xgb_model.predict(X_test)

# 2. Evaluate performance
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# 3. Save the model to your Drive
model_path = '/content/drive/MyDrive/Fraud_Detection_Project/xgboost_fraud_model.pkl'
joblib.dump(xgb_model, model_path)
print(f"\nModel saved successfully to: {model_path}")

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [14:25:46] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.94      0.96    113975
           1       0.32      0.82      0.46      4133

    accuracy                           0.93    118108
   macro avg       0.66      0.88      0.71    118108
weighted avg       0.97      0.93      0.95    118108

Confusion Matrix:
 [[106741   7234]
 [   734   3399]]

Model saved successfully to: /content/drive/MyDrive/Fraud_Detection_Project/xgboost_fraud_model.pkl


In [18]:
# @title 💳 AI Fraud Security Simulator { run: "auto" }

import plotly.graph_objects as go
import random

# UI Elements built natively into Colab
Scenario = "Test a Fraudulent Transaction" # @param ["Test a Normal Transaction", "Test a Fraudulent Transaction"]
Amount_Multiplier = 11.6 # @param {type:"slider", min:0.1, max:20.0, step:0.5}

# 1. Grab a random transaction
if Scenario == 'Test a Normal Transaction':
    sample_idx = y_test[y_test == 0].sample(1).index[0]
else:
    sample_idx = y_test[y_test == 1].sample(1).index[0]

sample_data = X_test.loc[[sample_idx]].copy()

# 2. Apply the multiplier to the amount
original_amt = sample_data['TransactionAmt'].values[0]
new_amt = original_amt * Amount_Multiplier
sample_data['TransactionAmt'] = new_amt

# 3. Predict the probability of fraud
fraud_prob = xgb_model.predict_proba(sample_data)[0][1] * 100

# 4. Render the chart
fig = go.Figure(go.Indicator(
    mode = "gauge+number",
    value = fraud_prob,
    domain = {'x': [0, 1], 'y': [0, 1]},
    title = {'text': f"AI Risk Score<br><span style='font-size:0.8em;color:gray'>Original: ${original_amt:.2f} | Modified: ${new_amt:.2f}</span>"},
    gauge = {
        'axis': {'range': [0, 100]},
        'bar': {'color': "black"},
        'steps': [
            {'range': [0, 30], 'color': "lightgreen"},
            {'range': [30, 70], 'color': "gold"},
            {'range': [70, 100], 'color': "crimson"}
        ],
        'threshold': {
            'line': {'color': "black", 'width': 4},
            'thickness': 0.75,
            'value': 70}
    }
))

fig.show()